# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** ML-04 (`w04_baseline_score.ipynb`) already
confirmed two signals — staleness→decline and CTR-vs-position — as inputs to the rule baseline.
This notebook audits three *different* safe signals plus one of FlyRank's real product-flag
assumptions, using the `auditing-signals` skill's verdict discipline (CONFIRMED / OPPOSITE /
MIXED / FALSE, sample-size floors, weighted rates not averaged rates).

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f'{len(df):,} rows, {df["client_id"].nunique()} clients')

for col in ['impressions_90d', 'word_count', 'ctr']:
    print(f'\n{col}:')
    print(df[col].describe().round(2))
    print(f'  skew (raw):        {df[col].skew():.2f}')
    if col != 'ctr':
        print(f'  skew (log1p):       {np.log1p(df[col]).skew():.2f}')

print(f"\nword_count missing: {df['word_count'].isna().sum():,} of {len(df):,} rows")

30,000 rows, 32 clients

impressions_90d:
count     30000.00
mean       5200.37
std       16838.02
min           1.00
25%          81.00
50%         731.00
75%        3615.25
max      517715.00
Name: impressions_90d, dtype: float64
  skew (raw):        11.38
  skew (log1p):       -0.39

word_count:
count    22301.00
mean      3107.76
std       1452.38
min          8.00
25%       2413.00
50%       2877.00
75%       3666.00
max       9546.00
Name: word_count, dtype: float64
  skew (raw):        0.94
  skew (log1p):       -0.61

ctr:
count    30000.00
mean         0.51
std          3.28
min          0.00
25%          0.00
50%          0.07
75%          0.29
max        100.00
Name: ctr, dtype: float64
  skew (raw):        17.44

word_count missing: 7,699 of 30,000 rows


**Read:** all three fields are heavy-tailed in the raw scale — `impressions_90d` has skew
11.4 (a handful of pages carry hundreds of thousands of impressions while the median is 731),
and `ctr` is even worse (skew 17.4, dominated by rare rows near 100%). `log1p` visibly tames
`impressions_90d` (skew drops to -0.39) and `word_count` (skew drops to -0.61), so every test
below uses **grouped medians/bucket tiers** or **weighted rates** rather than raw Pearson
correlation on the untransformed columns — exactly the trap this skill calls out. `word_count`
is also missing for a real chunk of rows (7,699 of 30,000) — per the data dictionary this tracks
`content_type`, not randomness, so any test using it states its own `n` rather than assuming
full coverage.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
FLOOR = 50  # per the skill: no verdict from a bucket under ~50 rows

# --- Signal test #1: 'older content decays more' ---
print('=== Signal 1: content age (age_tier) vs decline_rate ===')
t1 = (df.groupby('age_tier')
        .agg(n=('content_id', 'count'), decline_rate=('is_declining', 'mean'))
        .reindex(['0-14', '15-30', '31-90', '91-180', '181-365', '365+']))
print(t1.round(4))
print()
trusted1 = t1[t1['n'] >= FLOOR]
print(f'Trusted buckets (n >= {FLOOR}):')
print(trusted1.round(4))

=== Signal 1: content age (age_tier) vs decline_rate ===
                n  decline_rate
age_tier                       
0-14          NaN           NaN
15-30         NaN           NaN
31-90       492.0        0.6687
91-180    11780.0        0.6256
181-365   11368.0        0.5149
365+       6360.0        0.4263

Trusted buckets (n >= 50):
                n  decline_rate
age_tier                       
31-90       492.0        0.6687
91-180    11780.0        0.6256
181-365   11368.0        0.5149
365+       6360.0        0.4263


**Verdict: OPPOSITE.** The claim "older content decays more" does not hold — decline rate runs
the other way, from 66.9% at `31-90` days down to 42.6% at `365+` days, across four
well-powered buckets (`0-14` and `15-30` have 0 rows in this slice and can't be assessed).
In practice: the oldest surviving content in this dataset is the *most* stable, not the least —
likely survivorship (content that ranked poorly and got pruned early never reaches the `365+`
bucket at all). This is a genuinely different signal from staleness (`days_since_last_update`,
tested in ML-04): age since creation and time since last edit tell opposite stories here.

In [3]:
# --- Signal test #2: 'harder keywords churn more' ---
print('=== Signal 2: competition_level vs decline_rate ===')
t2 = (df.groupby('competition_level')
        .agg(n=('content_id', 'count'), decline_rate=('is_declining', 'mean'))
        .reindex(['LOW', 'MEDIUM', 'HIGH']))
print(t2.round(4))
print(f"\n(missing competition_level: {df['competition_level'].isna().sum():,} rows — no keyword data, per the dictionary)")

=== Signal 2: competition_level vs decline_rate ===
                       n  decline_rate
competition_level                     
LOW                22896        0.5632
MEDIUM              1836        0.5670
HIGH                2658        0.5561

(missing competition_level: 2,610 rows — no keyword data, per the dictionary)


**Verdict: FALSE.** Decline rate is essentially flat across competition tiers — 56.3% (LOW),
56.7% (MEDIUM), 55.6% (HIGH) — a spread under 1.2pp on three buckets that easily clear the
floor (n = 22,896 / 1,836 / 2,658). Keyword competition, at least measured this way, is not a
content-refresh signal worth building a rule on. Worth publishing as-is: a confirmed non-signal
stops the next person from wiring `competition_level` into a score and finding nothing there
the hard way.

In [4]:
# --- Signal test #3: 'shorter content converts clicks better, at a cost of eligible volume' ---
print('=== Signal 3: word_count_tier vs weighted CTR (clicks-sum / impressions-sum, NOT mean-of-CTR) ===')
t3 = (df.groupby('word_count_tier')
        .agg(n=('content_id', 'count'),
             clicks=('clicks_90d', 'sum'),
             impressions=('impressions_90d', 'sum'))
        .reindex(['<1000', '1000-2000', '2000-3500', '3500+']))
t3['weighted_ctr_pct'] = 100 * t3['clicks'] / t3['impressions']
print(t3.round(4))

=== Signal 3: word_count_tier vs weighted CTR (clicks-sum / impressions-sum, NOT mean-of-CTR) ===
                     n  clicks  impressions  weighted_ctr_pct
word_count_tier                                              
<1000              973     969        32071            3.0214
1000-2000         3780   13692      4663470            0.2936
2000-3500        11263  216836     62916991            0.3446
3500+             6285  131339     45645973            0.2877


**Verdict: MIXED.** The `<1000` tier shows a striking weighted CTR (3.02%) versus a flat
0.29-0.34% across every longer tier (n=973, well above the floor, and the ratio is weighted by
true click/impression sums, not an average-of-CTRs trap). But this tier also carries far less
total volume (32,071 impressions across 973 rows — roughly 33 impressions/row versus millions
of total impressions in the other tiers), which is exactly the kind of thin-denominator pattern
the skill warns about even after clearing the row-count floor. I'm calling this MIXED rather
than CONFIRMED: the direction is real and worth a closer look (short content may skew toward
branded/navigational queries that click at a naturally higher rate, not toward short content
being inherently better), but I'm not ready to act on it as "shorten your content" without
controlling for query intent first.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's
assumption?*

Testing **`thin_visible_page`**: `word_count > 0`, `word_count < 1200`, and
`impressions_90d >= 250` (per `docs/ml-intern-dataset-and-lane-guide.md`). The flag's implicit
assumption is that thin content underperforms and is worth flagging for a rewrite.

In [5]:
eligible = df[(df['word_count'] > 0) & (df['impressions_90d'] >= 250)].copy()
eligible['is_thin'] = eligible['word_count'] < 1200

flag_test = (eligible.groupby('is_thin')
               .agg(n=('content_id', 'count'),
                    decline_rate=('is_declining', 'mean'),
                    clicks=('clicks_90d', 'sum'),
                    impressions=('impressions_90d', 'sum')))
flag_test['weighted_ctr_pct'] = 100 * flag_test['clicks'] / flag_test['impressions']
print(flag_test.round(4))
print()
n_thin = int(eligible['is_thin'].sum())
print(f'thin_visible_page would fire on {n_thin} of {len(eligible):,} eligible rows in this slice.')

             n  decline_rate  clicks  impressions  weighted_ctr_pct
is_thin                                                            
False    13596        0.6467  359578    112684149            0.3191
True        82        0.5366    1250       129910            0.9622

thin_visible_page would fire on 82 of 13,678 eligible rows in this slice.


**Verdict: OPPOSITE — and worth flagging loudly.** Among rows eligible for `thin_visible_page`
(word count known, ≥250 impressions/90d), the 82 rows the flag would actually fire on show a
**lower** decline rate (53.7% vs 64.7%) and a **3x higher** weighted CTR (0.96% vs 0.32%) than
the non-thin population. That's the opposite of the flag's assumption that thin content is
underperforming. `n=82` clears the 50-row floor but is thin itself — I'd treat this as
"insufficient evidence to raise the alarm on its own" rather than "the flag is wrong," and I'd
want to see it hold on a second month before recommending anyone touch the flag's threshold.
Still, it's exactly the kind of result the `hunting-leakage-and-validating` mindset is for:
the flag encodes someone's hand-tuned assumption, and this slice doesn't support it — a finding
worth reporting honestly rather than smoothing over.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [6]:
print('Summary for a content team:')
print('1. Content age is NOT a refresh trigger on its own -- older pages are the stable ones')
print('   here, so age-based rules should point the other way from intuition, if at all.')
print('2. Keyword competition level carries no measurable signal in this slice -- do not spend')
print('   scoring budget on it.')
print('3. The thin_visible_page flag is not supported by this data -- word count under 1200')
print('   does not predict decline or low CTR here. Recommend re-checking that threshold on a')
print('   second month before trusting it in the baseline rule.')

Summary for a content team:
1. Content age is NOT a refresh trigger on its own -- older pages are the stable ones
   here, so age-based rules should point the other way from intuition, if at all.
2. Keyword competition level carries no measurable signal in this slice -- do not spend
   scoring budget on it.
3. The thin_visible_page flag is not supported by this data -- word count under 1200
   does not predict decline or low CTR here. Recommend re-checking that threshold on a
   second month before trusting it in the baseline rule.


A content team reading this should walk away with: staleness and CTR-vs-position (ML-04) are
the two signals that actually hold up and are worth ranking on; content age and keyword
competition are not, on this slice; and the `thin_visible_page` flag's own assumption doesn't
survive contact with the data, so it should be re-audited rather than trusted as-is before
it's folded into any scoring model — a flag that fires backwards would send review effort to
exactly the wrong pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.